# AI-Enhanced Simple Banking: Fraud Detection ML Pipeline

This notebook documents the reproducible machine learning pipeline for detecting fraudulent transactions in the simple banking application.

### Pipeline Overview:
1. **Synthetic Data Loading & EDA**: Understanding normal vs anomalous financial patterns.
2. **Data Preprocessing**: Handling missing values, scaling numeric features, and one-hot encoding transaction types.
3. **Model Training**: Baseline comparison (Weighted Logistic Regression) vs Primary (Balanced Random Forest Classifier).
4. **Evaluation**: Accuracy, Precision, Recall, F1-Score, Confusion Matrix, and ROC-AUC.
5. **Risk Classification**: Calibrating three risk categories (`NORMAL`, `SUSPICIOUS`, `HIGH_RISK`).

In [1]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

print("Libraries loaded successfully!")

## 1. Load Dataset & Explore Class Imbalance

In [2]:
data_path = os.path.join("..", "data", "synthetic_transactions.csv")
df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape}")
df.head()

In [3]:
# Class distribution analysis
print(df["is_fraud"].value_counts(normalize=True) * 100)
sns.countplot(data=df, x="is_fraud", palette=["#2563eb", "#dc2626"])
plt.title("Class Imbalance in Transactions (0=Normal, 1=Fraud)")
plt.show()

## 2. Feature Selection & Train/Test Split

In [4]:
numerical_features = [
    "amount_inr", "hour_of_day", "day_of_week", "is_weekend",
    "tx_frequency_last_24h", "balance_before_inr", "balance_ratio",
    "customer_hist_avg_inr", "amount_to_avg_ratio", "time_since_last_tx_min",
    "is_rapid_succession"
]
categorical_features = ["tx_type"]
feature_cols = numerical_features + categorical_features

X = df[feature_cols]
y = df["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set:  {X_test.shape[0]} samples")

## 3. Preprocessing Pipeline & Model Training

In [5]:
num_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", num_transformer, numerical_features),
    ("cat", cat_transformer, categorical_features)
])

rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        max_depth=12,
        min_samples_split=4,
        class_weight="balanced",
        random_state=42
    ))
])

rf_model.fit(X_train, y_train)
print("Random Forest Model trained successfully!")

## 4. Evaluation & Risk Scoring Thresholds

In [6]:
y_pred = rf_model.predict(X_test)
y_proba = rf_model.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}")

In [7]:
# Documenting Three Risk Categories:
# - NORMAL:     p < 0.25
# - SUSPICIOUS: 0.25 <= p < 0.70
# - HIGH_RISK:  p >= 0.70
def classify_risk(prob):
    if prob < 0.25:
        return "NORMAL"
    elif prob < 0.70:
        return "SUSPICIOUS"
    else:
        return "HIGH_RISK"

sample_probs = [0.05, 0.42, 0.89]
for p in sample_probs:
    print(f"Probability: {p:.2f} -> Risk Level: {classify_risk(p)}")